# 04 — Performance Analytics

**Bluestock Mutual Fund Analytics Capstone — D4**

This notebook calculates and validates the main fund-performance metrics required by the rubric:

- Annualized return / CAGR using **252 trading days**
- Annualized volatility / standard deviation
- Sharpe ratio
- Beta versus a benchmark
- Historical Value at Risk (VaR)
- Maximum drawdown
- Fund-level performance score
- Exportable performance metrics CSV

The calculations use the project's actual SQLite database and adapt to the available column names.


In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)


## 1. Load performance, NAV, fund and benchmark data

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'",
        conn
    )["name"].tolist()

def load_table(name):
    if name not in tables:
        return pd.DataFrame()
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(f'SELECT * FROM "{name}"', conn)

fund = load_table("dim_fund")
nav = load_table("fact_nav")
perf_existing = load_table("fact_performance")
benchmark = load_table("fact_benchmark")

print("Tables loaded:")
for name, df in {
    "dim_fund": fund,
    "fact_nav": nav,
    "fact_performance": perf_existing,
    "fact_benchmark": benchmark
}.items():
    print(f"  {name}: {df.shape}")


## 2. Inspect schemas

In [ ]:
for name, df in {
    "dim_fund": fund,
    "fact_nav": nav,
    "fact_performance": perf_existing,
    "fact_benchmark": benchmark
}.items():
    print(f"\n{name}")
    display(pd.DataFrame({"column": df.columns, "dtype": [str(x) for x in df.dtypes]}))


## 3. Prepare NAV returns

For daily return calculations:

\[
R_t = \frac{NAV_t}{NAV_{t-1}} - 1
\]

Returns are calculated separately for each fund after sorting by date.


In [ ]:
if nav.empty:
    raise ValueError("fact_nav is empty or missing.")

nav = nav.copy()

# Normalize common column names.
date_candidates = ["date", "nav_date", "as_of_date", "valuation_date"]
fund_candidates = ["amfi_code", "scheme_code", "fund_code"]
nav_candidates = ["nav", "nav_value", "net_asset_value"]

date_col = next((c for c in date_candidates if c in nav.columns), None)
fund_col = next((c for c in fund_candidates if c in nav.columns), None)
nav_col = next((c for c in nav_candidates if c in nav.columns), None)

if not all([date_col, fund_col, nav_col]):
    raise ValueError(
        f"Could not identify NAV columns. Found: {list(nav.columns)}"
    )

nav["date"] = pd.to_datetime(nav[date_col], errors="coerce")
nav["amfi_code"] = nav[fund_col]
nav["nav_value"] = pd.to_numeric(nav[nav_col], errors="coerce")

nav = (
    nav[["date", "amfi_code", "nav_value"]]
    .dropna(subset=["date", "amfi_code", "nav_value"])
    .sort_values(["amfi_code", "date"])
    .drop_duplicates(["amfi_code", "date"], keep="last")
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav_value"]
    .pct_change()
)

nav = nav.replace([np.inf, -np.inf], np.nan)

print("Prepared NAV rows:", len(nav))
display(nav.head())


## 4. Prepare benchmark returns

Beta is calculated from fund and benchmark returns over their overlapping observations:

\[
\beta_i = \frac{Cov(R_i,R_m)}{Var(R_m)}
\]

The benchmark is identified from the project's `fact_benchmark` table when a usable date/value pair is available.


In [ ]:
benchmark_return = pd.DataFrame()

if not benchmark.empty:
    bdate_candidates = ["date", "benchmark_date", "as_of_date"]
    bvalue_candidates = ["index_value", "benchmark_value", "value", "close", "nav"]

    bdate_col = next((c for c in bdate_candidates if c in benchmark.columns), None)
    bvalue_col = next((c for c in bvalue_candidates if c in benchmark.columns), None)

    if bdate_col and bvalue_col:
        benchmark_return = benchmark[[bdate_col, bvalue_col]].copy()
        benchmark_return.columns = ["date", "benchmark_value"]
        benchmark_return["date"] = pd.to_datetime(
            benchmark_return["date"], errors="coerce"
        )
        benchmark_return["benchmark_value"] = pd.to_numeric(
            benchmark_return["benchmark_value"], errors="coerce"
        )
        benchmark_return = (
            benchmark_return
            .dropna()
            .sort_values("date")
            .drop_duplicates("date", keep="last")
        )
        benchmark_return["benchmark_return"] = (
            benchmark_return["benchmark_value"].pct_change()
        )

print("Benchmark observations:", len(benchmark_return))
display(benchmark_return.head())


## 5. Metric formulas

### CAGR / annualized return

For a period containing `N` trading observations:

\[
CAGR = \left(\frac{V_{end}}{V_{start}}\right)^{252/N}-1
\]

### Annualized volatility

\[
\sigma_{annual} = \sigma_{daily}\sqrt{252}
\]

### Sharpe ratio

Using a configurable annual risk-free rate:

\[
Sharpe = \frac{R_{annualized}-R_f}{\sigma_{annualized}}
\]

### Historical VaR

At confidence level 95%, historical VaR is the negative of the 5th percentile of daily returns:

\[
VaR_{95\%} = -Q_{0.05}(R)
\]

A positive VaR therefore represents a loss magnitude.

### Maximum drawdown

\[
Drawdown_t = \frac{V_t}{\max_{s\le t}V_s}-1
\]


In [ ]:
TRADING_DAYS = 252
RISK_FREE_ANNUAL = 0.06
VAR_CONFIDENCE = 0.95

def annualized_cagr(values):
    values = pd.Series(values).dropna()
    if len(values) < 2 or values.iloc[0] <= 0 or values.iloc[-1] <= 0:
        return np.nan
    n_trading_days = len(values) - 1
    if n_trading_days <= 0:
        return np.nan
    return (values.iloc[-1] / values.iloc[0]) ** (TRADING_DAYS / n_trading_days) - 1


def annualized_volatility(returns):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2:
        return np.nan
    return returns.std(ddof=1) * np.sqrt(TRADING_DAYS)


def sharpe_ratio(returns, risk_free_annual=RISK_FREE_ANNUAL):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2:
        return np.nan
    annual_return = returns.mean() * TRADING_DAYS
    annual_vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    if annual_vol == 0:
        return np.nan
    return (annual_return - risk_free_annual) / annual_vol


def historical_var(returns, confidence=VAR_CONFIDENCE):
    returns = pd.Series(returns).dropna()
    if len(returns) < 20:
        return np.nan
    return -returns.quantile(1 - confidence)


def max_drawdown(values):
    values = pd.Series(values).dropna()
    if values.empty:
        return np.nan
    running_max = values.cummax()
    drawdown = values / running_max - 1
    return drawdown.min()


def beta_vs_benchmark(fund_returns, benchmark_returns):
    merged = pd.concat(
        [
            pd.Series(fund_returns, name="fund_return"),
            pd.Series(benchmark_returns, name="benchmark_return")
        ],
        axis=1
    ).dropna()

    if len(merged) < 20:
        return np.nan

    benchmark_variance = merged["benchmark_return"].var(ddof=1)
    if benchmark_variance == 0 or pd.isna(benchmark_variance):
        return np.nan

    return (
        merged["fund_return"].cov(merged["benchmark_return"])
        / benchmark_variance
    )


## 6. Calculate fund-level metrics

In [ ]:
records = []

benchmark_daily = (
    benchmark_return[["date", "benchmark_return"]].copy()
    if not benchmark_return.empty
    else pd.DataFrame(columns=["date", "benchmark_return"])
)

for amfi_code, group in nav.groupby("amfi_code"):
    group = group.sort_values("date").copy()
    returns = group["daily_return"].dropna()

    if len(group) < 2:
        continue

    cagr = annualized_cagr(group["nav_value"])
    volatility = annualized_volatility(returns)
    sharpe = sharpe_ratio(returns)
    var95 = historical_var(returns)
    mdd = max_drawdown(group["nav_value"])

    beta = np.nan

    if not benchmark_daily.empty:
        overlap = (
            group[["date", "daily_return"]]
            .rename(columns={"daily_return": "fund_return"})
            .merge(benchmark_daily, on="date", how="inner")
        )

        if len(overlap) >= 20:
            benchmark_var = overlap["benchmark_return"].var(ddof=1)
            if benchmark_var and not pd.isna(benchmark_var):
                beta = (
                    overlap["fund_return"].cov(overlap["benchmark_return"])
                    / benchmark_var
                )

    records.append({
        "amfi_code": amfi_code,
        "start_date": group["date"].min(),
        "end_date": group["date"].max(),
        "observations": len(group),
        "start_nav": group["nav_value"].iloc[0],
        "end_nav": group["nav_value"].iloc[-1],
        "cagr_pct": cagr * 100 if pd.notna(cagr) else np.nan,
        "annualized_volatility_pct": volatility * 100 if pd.notna(volatility) else np.nan,
        "sharpe_ratio": sharpe,
        "beta": beta,
        "historical_var_95_pct": var95 * 100 if pd.notna(var95) else np.nan,
        "max_drawdown_pct": mdd * 100 if pd.notna(mdd) else np.nan
    })

metrics = pd.DataFrame(records)

if not metrics.empty and not fund.empty and "amfi_code" in fund.columns:
    metrics = metrics.merge(
        fund,
        on="amfi_code",
        how="left"
    )

print("Funds with calculated metrics:", len(metrics))
display(metrics.head(10))


## 7. Quality checks for metric calculations

In [ ]:
quality_checks = pd.DataFrame({
    "metric": [
        "CAGR",
        "Annualized volatility",
        "Sharpe ratio",
        "Beta",
        "Historical VaR 95%",
        "Maximum drawdown"
    ],
    "missing_values": [
        int(metrics["cagr_pct"].isna().sum()),
        int(metrics["annualized_volatility_pct"].isna().sum()),
        int(metrics["sharpe_ratio"].isna().sum()),
        int(metrics["beta"].isna().sum()),
        int(metrics["historical_var_95_pct"].isna().sum()),
        int(metrics["max_drawdown_pct"].isna().sum())
    ]
})

display(quality_checks)

print("Metric sanity checks:")
print("CAGR finite:", np.isfinite(metrics["cagr_pct"].dropna()).all())
print("Volatility non-negative:", (metrics["annualized_volatility_pct"].dropna() >= 0).all())
print("VaR non-negative:", (metrics["historical_var_95_pct"].dropna() >= 0).all())
print("Max drawdown non-positive:", (metrics["max_drawdown_pct"].dropna() <= 0).all())


## 8. Performance score

In [ ]:
# A transparent score for ranking, not an investment recommendation.
# Higher return and Sharpe increase the score; higher volatility and drawdown reduce it.

score_df = metrics.copy()

def zscore(series):
    series = pd.to_numeric(series, errors="coerce")
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std

score_df["z_cagr"] = zscore(score_df["cagr_pct"])
score_df["z_sharpe"] = zscore(score_df["sharpe_ratio"])
score_df["z_volatility"] = zscore(score_df["annualized_volatility_pct"])
score_df["z_drawdown"] = zscore(score_df["max_drawdown_pct"])

score_df["performance_score"] = (
    0.40 * score_df["z_cagr"].fillna(0)
    + 0.30 * score_df["z_sharpe"].fillna(0)
    - 0.20 * score_df["z_volatility"].fillna(0)
    + 0.10 * score_df["z_drawdown"].fillna(0)
)

score_df = score_df.sort_values(
    "performance_score",
    ascending=False
)

display(
    score_df[
        [
            c for c in [
                "scheme_name",
                "fund_house",
                "category",
                "cagr_pct",
                "annualized_volatility_pct",
                "sharpe_ratio",
                "beta",
                "historical_var_95_pct",
                "max_drawdown_pct",
                "performance_score"
            ]
            if c in score_df.columns
        ]
    ].head(20)
)


## 9. Visual analysis — return vs risk

In [ ]:
plot_df = metrics.dropna(
    subset=["cagr_pct", "annualized_volatility_pct"]
).copy()

plt.figure(figsize=(10, 6))
plt.scatter(
    plot_df["annualized_volatility_pct"],
    plot_df["cagr_pct"],
    alpha=0.55
)
plt.xlabel("Annualized Volatility (%)")
plt.ylabel("CAGR (%)")
plt.title("Fund Risk vs Annualized Return")
plt.grid(alpha=0.25)
plt.show()


## 10. Visual analysis — Sharpe distribution

In [ ]:
plt.figure(figsize=(10, 5))
metrics["sharpe_ratio"].dropna().hist(bins=30)
plt.xlabel("Sharpe Ratio")
plt.ylabel("Number of Funds")
plt.title("Distribution of Sharpe Ratios")
plt.grid(alpha=0.25)
plt.show()


## 11. Top funds by performance score

In [ ]:
top_cols = [
    c for c in [
        "scheme_name",
        "fund_house",
        "category",
        "cagr_pct",
        "annualized_volatility_pct",
        "sharpe_ratio",
        "beta",
        "historical_var_95_pct",
        "max_drawdown_pct",
        "performance_score"
    ]
    if c in score_df.columns
]

display(score_df[top_cols].head(15))


## 12. Export D4 performance metrics

In [ ]:
output_path = PROCESSED_DIR / "fund_performance_metrics.csv"

export_df = score_df.copy()

for col in export_df.columns:
    if pd.api.types.is_datetime64_any_dtype(export_df[col]):
        export_df[col] = export_df[col].dt.strftime("%Y-%m-%d")

export_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(export_df):,}")


## 13. Interpretation and limitations

### Interpretation

- **CAGR** measures annualized growth over the observed trading period.
- **Volatility** measures dispersion of daily returns, annualized using √252.
- **Sharpe ratio** measures return above the configured annual risk-free rate per unit of annualized volatility.
- **Beta** measures sensitivity to the selected benchmark over overlapping daily observations.
- **Historical VaR at 95%** estimates the loss threshold exceeded by roughly the worst 5% of historical daily returns.
- **Maximum drawdown** measures the largest peak-to-trough decline in NAV.
- The performance score is a transparent ranking aid and **not investment advice**.

### Important limitations

- Historical metrics do not guarantee future performance.
- VaR depends on the historical return distribution and does not describe losses beyond the VaR threshold.
- Beta requires sufficient overlapping fund and benchmark observations.
- A 6% annual risk-free rate is used as a configurable analytical assumption; replace it with the rate specified by the project rubric/source if a different assumption is required.
- CAGR uses **252 trading days**, as required by the rubric, rather than calendar days.
